# 01 - RAG retrieval, step by step

**Goal:** follow the course labs 10 to 16 on the project's index: the Qdrant collection, then
one question searched in dense, BM25 and hybrid mode, with query rewriting and with reranking.
Everything goes through `src/rag/` (the code the agent uses); nothing is written to Qdrant.
Needs Qdrant and Ollama running (`docker compose up -d`).

In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
ROOT = Path.cwd().resolve().parents[1]  # notebooks/<module>/ -> repository root
for path in (ROOT, ROOT / "backend"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

import logging
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

plt.rcParams.update({"figure.dpi": 72, "savefig.dpi": 72})
%config InlineBackend.figure_formats = ["jpeg"]
pd.set_option("display.max_colwidth", 90)


def fmt(frame, digits=3, **columns):
    """Format the float columns with f"{x:.{digits}f}", the rounding of the result tables."""
    out = frame.copy()
    for name in out.columns:
        if out[name].dtype.kind == "f":
            d = columns.get(name, digits)
            out[name] = out[name].map(lambda v, d=d: "" if pd.isna(v) else f"{v:.{d}f}")
    return out


def show_image(path, title="", width=480):
    """Display an image file, downscaled (the notebook stays small)."""
    import io

    from IPython.display import Image as JpegImage

    image = Image.open(path).convert("RGB")
    image.thumbnail((width, width))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=80)
    if title:
        display(Markdown(f"*{title}*"))
    display(JpegImage(data=buffer.getvalue(), format="jpeg"))

## 1. The collection (lab 10)

One point per chunk: a dense bge-m3 vector, a BM25 sparse vector and the chunk's metadata.

In [2]:
from qdrant_client import QdrantClient

from app.config import Settings

settings = Settings()
qdrant = QdrantClient(url=settings.qdrant_url)
info = qdrant.get_collection(settings.qdrant_collection)
print(f"collection {settings.qdrant_collection}: {info.points_count} points")
print(
    "dense vector:", {k: (v.size, str(v.distance)) for k, v in info.config.params.vectors.items()}
)
print("sparse vectors:", list(info.config.params.sparse_vectors))
point = qdrant.scroll(settings.qdrant_collection, limit=1, with_payload=True)[0][0]
pd.Series(
    {k: str(v)[:80] for k, v in point.payload.items()}, name="payload of one point"
).to_frame()

collection tujya_chunks_v2: 27302 points
dense vector: {'dense': (1024, 'Cosine')}
sparse vectors: ['bm25']


,payload of one point
chunk_id,593aee3a2ba905c8
doc_id,ddinter:DDInter771:DDInter1662
doc_type,interaction
ingredients,"['folic acid', 'sertraline']"
mentioned_ingredients,[]
section,interaction
chunk_index,0
title,DDInter 2.0: folic acid + sertraline (Unknown)
text,DDInter 2.0 lists an interaction whose severity DDInter does not grade (level Un
source_name,DDInter 2.0


## 2. Retriever (lab 11)

The agent's retriever: Qdrant store, bge-m3 embeddings from Ollama, BM25 sparse encoder and
cross-encoder reranker. The fastembed models are cached in a temporary folder.

In [3]:
from src.rag.init_qdrant import OllamaEmbedder
from src.rag.reranking import CrossEncoderReranker
from src.rag.retrieval import Filters, Retriever
from src.rag.stores import open_store
from src.rag.stores.sparse import FastEmbedBM25

cache = Path(tempfile.gettempdir()) / "tujya-fastembed"
for name in ("httpx", "httpcore", "huggingface_hub"):
    logging.getLogger(name).setLevel(logging.WARNING)
embedder = OllamaEmbedder(
    settings.ollama_url,
    settings.embedding_model,
    settings.embedding_dim,
    num_ctx=settings.embedding_num_ctx,
)
retriever = Retriever(
    open_store(settings, embedder.dim),
    embedder,
    sparse=FastEmbedBM25(settings.rag_sparse_model, cache_dir=cache),
    reranker=CrossEncoderReranker(settings.rag_rerank_model, cache_dir=cache),
)
print(f"{len(retriever.chunks)} chunks loaded, BM25 runs in: {retriever.lexical_kind}")

27302 chunks loaded, BM25 runs in: qdrant-sparse:Qdrant/bm25


## 3. One question, three modes (labs 14 and 15)

The ingredients named in the question become a metadata filter, as the agent does.

In [4]:
QUESTION = "La warfarine interagit-elle avec le fluconazole ?"
FILTERS = Filters(ingredients=("warfarin", "fluconazole"))


def table(hits):
    return pd.DataFrame(
        [
            {"score": round(h.score, 4), "type": h.chunk.doc_type, "title": h.chunk.title[:70]}
            for h in hits
        ]
    )


async def search(query, mode, rerank=False):
    return await retriever.search(query, k=3, mode=mode, filters=FILTERS, rerank=rerank)


for mode in ("dense", "bm25", "hybrid"):
    display(Markdown(f"**{mode}**"))
    display(table(await search(QUESTION, mode)))

**dense**

,score,type,title
0,0.0164,label,Fluconazole label (Fluconazole): Drug interactions
1,0.0161,label,Warfarin label (Warfarin Sodium): Drug interactions
2,0.0159,label,Fluconazole label (Fluconazole): Precautions


**bm25**

,score,type,title
0,0.0164,interaction,DDInter 2.0: fluconazole + warfarin (Major)
1,0.0161,label,Fluconazole label (Fluconazole): Precautions
2,0.0159,label,Fluconazole label (Fluconazole): Drug interactions


**hybrid**

,score,type,title
0,0.0323,label,Fluconazole label (Fluconazole): Drug interactions
1,0.0320,interaction,DDInter 2.0: fluconazole + warfarin (Major)
2,0.0320,label,Fluconazole label (Fluconazole): Precautions


## 4. Query rewriting (lab 13)

The corpus is in English and BM25 only matches English words: the LLM rewrites the French
question into a short English query (`src/rag/query_rewriting.py`).

In [5]:
from src.agent.runtime import ollama_llm
from src.rag.query_rewriting import QueryRewriter

rewritten = await QueryRewriter(ollama_llm(settings)).rewrite(QUESTION)
print("rewritten query:", rewritten)
display(table(await search(rewritten, "hybrid")))

rewritten query: warfarine interacts fluconazole PubMed


,score,type,title
0,0.0325,label,Fluconazole label (Fluconazole): Drug interactions
1,0.0323,interaction,DDInter 2.0: fluconazole + warfarin (Major)
2,0.0320,label,Fluconazole label (Fluconazole): Precautions


## 5. Reranking (lab 16)

A cross-encoder reads the question with each candidate and reorders them. It is more precise
but costs seconds per question on the CPU, so the app keeps it off by default.

In [6]:
display(table(await search(rewritten, "hybrid", rerank=True)))

,score,type,title
0,5.0827,label,Warfarin label (Warfarin Sodium): Drug interactions
1,5.0646,label,Fluconazole label (Fluconazole): Drug interactions
2,4.3695,interaction,DDInter 2.0: fluconazole + warfarin (Major)
